# Bootstrap role stability

Does each node retain its original descriptive role when isolates are resampled within sources? This notebook runs the full 1,000 x 500 bootstrap analysis and recreates Figure 4b. It is not a compositional-null test. All runtime inputs are in the adjacent inputs/ folder; no original notebooks, project helpers, or network downloads are needed. Use sklearn-env. The baseline labels come from the original consensus graph; the bootstrap uses the verified independent-inner-source implementation.

Run sections 1-5 for a full analysis. For plotting only after a completed run, execute configuration, scientific definitions, input verification, and section 5. All 38 original nodes are shown, including human. Checkpoints cannot substitute for missing or failed replicates.

## 1. Configuration and definitions

In [ ]:
from pathlib import Path
from collections import Counter
import hashlib, json, pickle, time, platform, os, traceback, inspect
import numpy as np
import pandas as pd
import networkx as nx
import igraph as ig
import leidenalg, scipy, sklearn, joblib
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, Markdown, HTML, Image as DisplayImage
import matplotlib.pyplot as plt
from PIL import Image
from joblib import Parallel, delayed, parallel_config

CONFIG=dict(outer=1000, inner=500, pilot=20, seed=123, baseline_seed=186,
            cosine=.35, support=.50, partition=.50, role_quantile=.75)
WORKERS = 4  # Independent process workers; does not change replicate random streams.
BASE=Path.cwd()
INPUT=BASE/'inputs'
assert INPUT.is_dir(), 'Run from the notebook folder with its adjacent inputs/ directory.'
META=json.loads((INPUT/'metadata.json').read_text())
CLASSES=META['classes']; K=META['k']; METRICS=META['metrics']; ROLES=META['roles']
NODE_ORDER=META['order']; BASELINE_ROLES=META['baseline_roles']
INNER=CONFIG['inner']
PLOT=dict(font_family='Arial', node_fontsize=15, column_fontsize=15,
          colorbar_fontsize=15, panel_fontsize=22, annotation_fontsize=7,
          node_labels=META['node_labels'].copy(),
          row_role_order=['Global Connector','Broker/Bridge','Core Influencer','Peripheral/Local'],
          column_labels=['GC','B/B','CI','P/L','NR'], colorbar_label='Replicates (%)',
          panel_letter='', label_rotation=0, annotate=False,
          width_inches=5.2, aspect_ratio=1544/3515, dpi=300,
          axes=[.055,.055,.50,.887], colorbar_axes=[.055,.95,.50,.015],
          label_pad=5, role_colors=META['role_colors'].copy())


In [ ]:
def original_consensus(vectors, classes, k, seed, inner=None):
    """Preserve pandas.sample seed semantics and original Leiden vertex handling."""
    inner = CONFIG['inner'] if inner is None else inner
    size = len(classes)
    edge_counts = np.zeros((size,size),dtype=np.int32)
    partition_counts = np.zeros_like(edge_counts)
    rng = np.random.Generator(np.random.MT19937(seed))
    for iteration in range(inner):
        iter_seed = int(rng.integers(0,2**32,dtype=np.uint32))
        counts = np.zeros((size,k),dtype=float)
        for i,v in enumerate(vectors):
            # Original pandas sample creates a fresh RandomState with this same seed for every source.
            indices = np.random.RandomState(iter_seed).choice(len(v),size=len(v),replace=True)
            counts[i] = np.bincount(v[indices],minlength=k)
        sim = cosine_similarity(counts)
        mask = sim > CONFIG['cosine']
        np.fill_diagonal(mask,False)
        edge_counts += mask
        pairs = list(zip(*np.where(np.triu(mask,1))))
        # The original igraph constructor infers vertices from edges. Do not silently fix it here.
        graph = ig.Graph(edges=pairs,directed=False)
        graph.es['weight'] = [float(sim[i,j]) for i,j in pairs]
        part = leidenalg.find_partition(graph,leidenalg.ModularityVertexPartition,
                                       weights='weight',seed=int(seed))
        membership = np.asarray(part.membership)
        m = len(membership)
        partition_counts[:m,:m] += membership[:,None]==membership[None,:]
    support = edge_counts/inner
    coassign = partition_counts/inner
    np.fill_diagonal(coassign,1.)
    labels = fcluster(linkage(squareform(1-coassign),method='average'),
                      t=1-CONFIG['partition'],criterion='distance')
    frequencies = Counter(labels)
    retained = [classes[i] for i,c in enumerate(labels) if frequencies[c]>1]
    graph = nx.Graph()
    graph.add_nodes_from(classes)
    for i in range(size):
        for j in range(i+1,size):
            if support[i,j]>=CONFIG['support']:
                graph.add_edge(classes[i],classes[j],weight=float(support[i,j]))
    graph.remove_nodes_from(set(classes)-set(retained))
    return graph,dict(zip(classes,map(int,labels)))

def independent_consensus(counts, classes, seed):
    """Independent within-source multinomial resampling; no isolate-order dependence."""
    streams = [np.random.default_rng(s) for s in np.random.SeedSequence(seed).spawn(len(classes))]
    sizes = counts.sum(axis=1).astype(int)
    supports = [np.flatnonzero(row) for row in counts]
    probabilities = [row[idx] / n for row, idx, n in zip(counts, supports, sizes)]
    edge_count = np.zeros((len(classes), len(classes)), dtype=int)
    partition_count = np.zeros_like(edge_count)
    for iteration in range(INNER):
        sample = np.zeros_like(counts, dtype=float)
        for i, (rng, n, idx, p) in enumerate(zip(streams, sizes, supports, probabilities)):
            sample[i, idx] = rng.multinomial(n, p)
        sim = cosine_similarity(sample)
        mask = sim > 0.35
        np.fill_diagonal(mask, False)
        edge_count += mask
        pairs = list(zip(*np.where(np.triu(mask, 1))))
        graph = ig.Graph(n=len(classes), edges=pairs, directed=False)
        graph.es['weight'] = [float(sim[i, j]) for i, j in pairs]
        part = leidenalg.find_partition(graph, leidenalg.ModularityVertexPartition, weights='weight', seed=seed)
        labels = np.asarray(part.membership)
        partition_count += labels[:, None] == labels[None, :]
    coassign = partition_count / INNER
    np.fill_diagonal(coassign, 1.0)
    labels = fcluster(linkage(squareform(1 - coassign), method='average'), t=0.5, criterion='distance')
    frequencies = Counter(labels)
    retained = {classes[i] for i, label in enumerate(labels) if frequencies[label] > 1}
    graph = nx.Graph()
    graph.add_nodes_from(classes)
    for i in range(len(classes)):
        for j in range(i + 1, len(classes)):
            support = edge_count[i, j] / INNER
            if support >= 0.5:
                graph.add_edge(classes[i], classes[j], weight=float(support))
    graph.remove_nodes_from(set(classes) - retained)
    return graph

def node_metrics_weighted(G, weighted=True, degree_normalize=True):
    """
    Centrality metrics for a weighted similarity network.

    If weighted=True:
      - degree = normalized count of neighbors
      - strength = normalized sum of 'weight' edges
      - betweenness/closeness use a 'distance' attribute = 1 / weight
      - eigenvector and clustering use 'weight' directly.

    If weighted=False:
      - all metrics are computed on the unweighted graph.
    """
    if G.number_of_nodes() == 0 or G.number_of_edges() == 0:
        return pd.DataFrame()
    n = G.number_of_nodes()
    degree = dict(G.degree())
    strength = dict(G.degree(weight='weight'))
    if degree_normalize:
        degree = {k: v / (n - 1) for k, v in degree.items()}
        max_strength = max(strength.values()) if strength else 1.0
        strength = {k: v / max_strength for k, v in strength.items()}
    if weighted:
        H = G.copy()
        for u, v, d in H.edges(data=True):
            w = d.get('weight', 0.0)
            d['distance'] = 1.0 / w if w > 0 else np.inf
        clustering = nx.clustering(G, weight='weight')
        try:
            eigenvector = nx.eigenvector_centrality(G, weight='weight', max_iter=500)
        except nx.PowerIterationFailedConvergence:
            eigenvector = {node: np.nan for node in G.nodes()}
        betweenness = nx.betweenness_centrality(H, weight='distance', normalized=True)
        closeness = nx.closeness_centrality(H, distance='distance')
    else:
        clustering = nx.clustering(G)
        try:
            eigenvector = nx.eigenvector_centrality(G, max_iter=500)
        except nx.PowerIterationFailedConvergence:
            eigenvector = {node: np.nan for node in G.nodes()}
        betweenness = nx.betweenness_centrality(G, normalized=True)
        closeness = nx.closeness_centrality(G)
    df = pd.DataFrame({'degree': pd.Series(degree), 'strength': pd.Series(strength), 'clustering': pd.Series(clustering), 'betweenness': pd.Series(betweenness), 'closeness': pd.Series(closeness), 'eigenvector': pd.Series(eigenvector)})
    return df

def classify(metrics):
    if metrics.empty or not np.isfinite(metrics[METRICS].to_numpy()).all():
        raise RuntimeError('Undefined or non-finite centralities; no replicate is discarded.')
    e=float(np.percentile(metrics.eigenvector,75)); b=float(np.percentile(metrics.betweenness,75))
    he=metrics.eigenvector.ge(e); hb=metrics.betweenness.ge(b)
    roles=pd.Series(np.select([he&hb,hb&~he,he&~hb],ROLES[:3],default=ROLES[3]),index=metrics.index)
    return roles,dict(eigenvector=e,betweenness=b)

def atomic_pickle(path, value):
    tmp=path.with_name(path.name+f'.{os.getpid()}.tmp')
    tmp.write_bytes(pickle.dumps(value,protocol=5)); tmp.replace(path)

def edge_map(graph):
    return {tuple(sorted((a,b))):float(d['weight']) for a,b,d in graph.edges(data=True)}

def compute_replicate(index, use_checkpoint=True):
    path=RUN/'replicates'/f'{index:04d}.pkl'
    if use_checkpoint and path.exists():
        result=pickle.loads(path.read_bytes())
        assert result['fingerprint']==FINGERPRINT and result['index']==index
        return result
    start=time.monotonic()
    try:
        streams=np.random.SeedSequence([CONFIG['seed'],1,index,0]).spawn(len(vectors))
        outer=[v[np.random.default_rng(s).integers(0,len(v),len(v))] for v,s in zip(vectors,streams)]
        matrix=np.stack([np.bincount(v,minlength=K) for v in outer])
        assert np.array_equal(matrix.sum(axis=1),[len(v) for v in vectors])
        seed=int(np.random.SeedSequence([CONFIG['seed'],1,index]).generate_state(1)[0] % (2**31-1))
        graph=independent_consensus(matrix,CLASSES,seed)
        metrics=node_metrics_weighted(graph)[METRICS]
        roles,thresholds=classify(metrics)
        ranks=metrics.rank(method='average',pct=True)*100
        all_table=metrics.join(ranks.add_suffix('_percentile'))
        all_table['role']=roles
        table=all_table.reindex(NODE_ORDER)
        table['role']=table['role'].fillna('Not retained')
        assert table.loc[table.role.eq('Not retained'),METRICS].isna().all().all()
        result=dict(fingerprint=FINGERPRINT,index=index,table=table,all_retained=all_table,
                    thresholds=thresholds,outer_counts_sha256=hashlib.sha256(matrix.tobytes()).hexdigest(),
                    graph_nodes=len(graph),graph_edges=graph.number_of_edges(),seconds=time.monotonic()-start)
        if use_checkpoint: atomic_pickle(path,result)
        return result
    except Exception:
        (RUN/'failures'/f'{index:04d}.txt').write_text(traceback.format_exc())
        raise

def run_indices(indices):
    indices=list(indices)
    with parallel_config(backend='loky',inner_max_num_threads=1):
        results=Parallel(n_jobs=WORKERS,return_as='generator')(
            delayed(compute_replicate)(i) for i in indices)
        for j,result in enumerate(results,1):
            if j==1 or j%20==0 or j==len(indices):
                print(f'Completed/resumed {j}/{len(indices)}; replicate {result["index"]}',flush=True)


## 2. Input and baseline verification
Original baseline reproduction and the independent-stream check are separate, so implementation differences cannot silently redefine descriptive roles.

In [ ]:
manifest=json.loads((INPUT/'manifest.json').read_text())
for name,expected in manifest['files'].items():
    assert hashlib.sha256((INPUT/name).read_bytes()).hexdigest()==expected,name
archive=np.load(INPUT/'ordered_vectors.npz',allow_pickle=False)
vectors=[archive[f'v{i}'] for i in range(len(CLASSES))]
assert len(vectors)==59 and sum(map(len,vectors))==125582 and K==5354
assert len(NODE_ORDER)==38 and len(set(NODE_ORDER))==38 and set(NODE_ORDER)==set(BASELINE_ROLES)
from importlib.metadata import version as distribution_version
distributions={'numpy':'numpy','pandas':'pandas','networkx':'networkx',
               'igraph':'igraph','leidenalg':'leidenalg','scipy':'scipy',
               'sklearn':'scikit-learn','joblib':'joblib'}
versions={name:distribution_version(package) for name,package in distributions.items()}
versions['python']=platform.python_version()
notebook=json.loads((BASE/'centrality_role_stability.ipynb').read_text())
engine_source=''.join(next(c['source'] for c in notebook['cells'] if c.get('metadata',{}).get('tag')=='scientific_engine'))
scientific_files=['ordered_vectors.npz','metadata.json']
identity=dict(config=CONFIG,code=hashlib.sha256(engine_source.encode()).hexdigest(),versions=versions,
              inputs={f:manifest['files'][f] for f in scientific_files})
FINGERPRINT=hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()
RUN=BASE/'outputs'/FINGERPRINT[:16]
for folder in [RUN,RUN/'replicates',RUN/'failures',RUN/'figures',RUN/'tables']:
    folder.mkdir(parents=True,exist_ok=True)
(RUN/'manifest.json').write_text(json.dumps(identity,indent=2))
# Conda/pip metadata can disagree with the imported module; verify both on resume.
runtime_versions={m.__name__:getattr(m,'__version__',versions[m.__name__])
                  for m in [np,pd,nx,ig,leidenalg,scipy,sklearn,joblib]}
runtime_versions['python']=platform.python_version()
runtime_identity=dict(module_versions=runtime_versions,distribution_versions=versions)
runtime_identity['fingerprint']=hashlib.sha256(json.dumps(runtime_identity,sort_keys=True).encode()).hexdigest()
runtime_file=RUN/'runtime_identity.json'
if runtime_file.exists():
    assert json.loads(runtime_file.read_text())==runtime_identity, 'Imported runtime changed; do not reuse these checkpoints.'
else:
    runtime_file.write_text(json.dumps(runtime_identity,indent=2))
version_disagreements={k:dict(imported=v,distribution=versions[k]) for k,v in runtime_versions.items() if v!=versions[k]}
if version_disagreements:
    print('Recorded imported/distribution version differences:',version_disagreements)

input_hash_before={name:hashlib.sha256((INPUT/name).read_bytes()).hexdigest() for name in manifest['files']}
g,partition=original_consensus(vectors,CLASSES,K,CONFIG['baseline_seed'])
expected=nx.Graph();expected.add_nodes_from(META['baseline_nodes'])
expected.add_weighted_edges_from(META['baseline_edges'])
assert set(g)==set(expected) and edge_map(g)==edge_map(expected)
metrics=node_metrics_weighted(g)[METRICS]
reference=pd.DataFrame.from_dict(META['baseline_metrics'],orient='index')[METRICS]
assert np.allclose(metrics.reindex(reference.index),reference,rtol=1e-8,atol=1e-10)
roles,thresholds=classify(metrics)
assert roles.to_dict()==BASELINE_ROLES
baseline_report=dict(passed=True,nodes=len(g),edges=g.number_of_edges(),thresholds=thresholds)
matrix=np.stack([np.bincount(v,minlength=K) for v in vectors])
g_ind=independent_consensus(matrix,CLASSES,CONFIG['baseline_seed'])
m_ind=node_metrics_weighted(g_ind)[METRICS];r_ind,t_ind=classify(m_ind)
baseline_report['independent_stream_check']=dict(nodes=len(g_ind),edges=g_ind.number_of_edges(),thresholds=t_ind,
    changed_original_labels={n:[BASELINE_ROLES[n],r_ind.get(n,'Not retained')] for n in NODE_ORDER if r_ind.get(n)!=BASELINE_ROLES[n]})
(RUN/'baseline_verification.json').write_text(json.dumps(baseline_report,indent=2))
print(json.dumps(baseline_report,indent=2))
display(Markdown('The original graph and role labels were reproduced separately from the independent-source implementation check. Baseline labels are not redefined by that check.'))


## 3. Twenty-replicate computational pilot
Valid pilot replicates are retained in the fixed 1,000-replicate run. No settings depend on observed role frequencies.

In [ ]:
pilot_cached_before=sum((RUN/'replicates'/f'{i:04d}.pkl').exists() for i in range(CONFIG['pilot']))
start=time.monotonic()
run_indices(range(CONFIG['pilot']))
pilot=[pickle.loads((RUN/'replicates'/f'{i:04d}.pkl').read_bytes()) for i in range(CONFIG['pilot'])]
import resource
pilot_report=dict(replicates=len(pilot),wall_seconds=time.monotonic()-start,
                  median_compute_seconds=float(np.median([r['seconds'] for r in pilot])),
                  parent_peak_rss_bytes=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
                  worker_peak_rss_bytes=resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss,
                  memory_note='macOS resource peak RSS; parent and completed child processes, not summed concurrent usage',workers=WORKERS)
pilot_report['fresh_replicates']=CONFIG['pilot']-pilot_cached_before
pilot_report['resumed_replicates']=pilot_cached_before
pilot_report['estimated_total_wall_minutes']=pilot_report['median_compute_seconds']*CONFIG['outer']/WORKERS/60
pilot_report['runtime_note']='Compute-time estimate excludes process startup and assumes four-worker throughput; cache-read time is not a fresh-run estimate.'
(RUN/'pilot.json').write_text(json.dumps(pilot_report,indent=2))
print(json.dumps(pilot_report,indent=2))


## 4. Full run and verification

In [ ]:
run_indices(range(CONFIG['pilot'],CONFIG['outer']))
results=[pickle.loads((RUN/'replicates'/f'{i:04d}.pkl').read_bytes()) for i in range(CONFIG['outer'])]
assert len(results)==1000 and all(r['fingerprint']==FINGERPRINT for r in results)
data=pd.concat([r['table'].rename_axis('node').reset_index().assign(replicate=r['index']) for r in results],ignore_index=True)
assert len(data)==38000 and not data.duplicated(['node','replicate']).any()
assert data.groupby('node').size().eq(1000).all()
assert data.role.isin(ROLES).all()
valid=data.role.ne('Not retained')
assert np.isfinite(data.loc[valid,METRICS].to_numpy()).all()
assert data.loc[~valid,METRICS].isna().all().all()
for i in [0,499,999]:
    fresh=compute_replicate(i,use_checkpoint=False)
    pd.testing.assert_frame_equal(fresh['table'],results[i]['table'],check_exact=False,rtol=1e-12,atol=1e-14)
    assert fresh['table']['role'].equals(results[i]['table']['role'])
    assert fresh['outer_counts_sha256']==results[i]['outer_counts_sha256']
first=RUN/'replicates'/'0000.pkl';before=(first.stat().st_mtime_ns,hashlib.sha256(first.read_bytes()).hexdigest())
compute_replicate(0)
assert before==(first.stat().st_mtime_ns,hashlib.sha256(first.read_bytes()).hexdigest())
assert input_hash_before=={name:hashlib.sha256((INPUT/name).read_bytes()).hexdigest() for name in manifest['files']}
data.to_csv(RUN/'tables'/'replicate_outcomes.csv.gz',index=False)
pd.concat([r['all_retained'].rename_axis('node').reset_index().assign(replicate=r['index']) for r in results]).to_csv(RUN/'tables'/'all_retained_metrics.csv.gz',index=False)
pd.DataFrame([dict(replicate=r['index'],graph_nodes=r['graph_nodes'],graph_edges=r['graph_edges'],**r['thresholds']) for r in results]).to_csv(RUN/'tables'/'replicate_thresholds.csv',index=False)
print('All 1,000 replicates complete; deterministic reruns, input preservation, and resume checks passed.')


## 5. Tables and figures
Change PLOT settings in section 1 to adjust all labels, fonts, margins, annotations, or aspect ratio. Scientific checkpoints are independent of plotting settings. The full-figure assembly checks the legacy panel ratio; custom-ratio standalone exports are not silently distorted to fit.

In [ ]:
if 'data' not in globals():
    data=pd.read_csv(RUN/'tables'/'replicate_outcomes.csv.gz')
frequencies=pd.crosstab(data.node,data.role).reindex(index=NODE_ORDER,columns=ROLES,fill_value=0)/CONFIG['outer']*100
assert frequencies.shape==(38,5) and np.allclose(frequencies.sum(axis=1),100)
mcse=100*np.sqrt((frequencies/100)*(1-frequencies/100)/CONFIG['outer'])
frequencies.to_csv(RUN/'tables'/'role_frequencies_percent.csv',index_label='node')
mcse.to_csv(RUN/'tables'/'role_frequency_mcse_percentage_points.csv',index_label='node')
recovery=pd.DataFrame([dict(node=n,baseline_role=BASELINE_ROLES[n],recovery_percent=frequencies.loc[n,BASELINE_ROLES[n]],retention_percent=100-frequencies.loc[n,'Not retained']) for n in NODE_ORDER])
recovery.to_csv(RUN/'tables'/'role_recovery.csv',index=False)
rows=[]
for node in NODE_ORDER:
    for metric in METRICS:
        values=data.loc[data.node.eq(node),metric].dropna()
        ranks=data.loc[data.node.eq(node),metric+'_percentile'].dropna()
        rows.append(dict(node=node,metric=metric,retained_replicates=len(values),raw_lower=values.quantile(.025),raw_median=values.median(),raw_upper=values.quantile(.975),percentile_lower=ranks.quantile(.025),percentile_median=ranks.median(),percentile_upper=ranks.quantile(.975)))
pd.DataFrame(rows).to_csv(RUN/'tables'/'centrality_summary.csv',index=False)
display(HTML(frequencies.to_html(float_format=lambda x:f'{x:.1f}')))
display(Markdown('Each row is one original node (source). Global Connector, Broker/Bridge, Core Influencer, and Peripheral/Local are percentages assigned those roles; Not retained is the percentage removed by community filtering. Every percentage uses all 1,000 outer replicates and each row sums to 100%. These are bootstrap frequencies, not probabilities that biological roles are true.'))
display(HTML(recovery.to_html(index=False,float_format=lambda x:f'{x:.1f}')))
display(Markdown('Each row is an original node. node is its source identifier; baseline_role is its original descriptive classification; recovery_percent is the percentage of all 1,000 replicates retaining that classification; retention_percent is the percentage of all replicates surviving community filtering, regardless of role.'))


In [ ]:
def plot_roles(style, name):
    # Reorder only the display; scientific input and checkpoint order stay unchanged.
    plot_nodes=[n for role in style['row_role_order'] for n in NODE_ORDER
                if BASELINE_ROLES[n] == role]
    assert len(plot_nodes)==len(NODE_ORDER) and set(plot_nodes)==set(NODE_ORDER)
    values=frequencies.reindex(index=plot_nodes, columns=ROLES)
    assert values.notna().all().all()
    width=style['width_inches'];height=width/style['aspect_ratio']
    with plt.rc_context({'font.family':style['font_family'],'svg.fonttype':'none'}):
        fig=plt.figure(figsize=(width,height),facecolor='white')
        ax=fig.add_axes(style['axes'])
        im=ax.imshow(values.to_numpy(),aspect='auto',cmap='viridis_r',vmin=0,vmax=100)
        ax.set_xticks(range(5),style['column_labels'],rotation=style['label_rotation'],ha='center' if style['label_rotation']==0 else 'right',fontsize=style['column_fontsize'])
        ax.set_yticks(range(38),[style['node_labels'].get(n,n) for n in plot_nodes],fontsize=style['node_fontsize'],fontweight='bold')
        ax.yaxis.tick_right();ax.tick_params(length=0,pad=style['label_pad'])
        for label,node in zip(ax.get_yticklabels(),plot_nodes): label.set_color(style['role_colors'][BASELINE_ROLES[node]])
        ax.set_xticks(np.arange(-.5,5,1),minor=True);ax.set_yticks(np.arange(-.5,38,1),minor=True)
        ax.grid(which='minor',color='white',linewidth=.4);ax.tick_params(which='minor',length=0)
        for spine in ax.spines.values():spine.set_visible(False)
        if style['annotate']:
            for row in range(38):
                for col in range(5):
                    value=values.iloc[row,col]
                    ax.text(col,row,f'{value:.1f}',ha='center',va='center',fontsize=style['annotation_fontsize'],color='white' if value>65 else 'black')
        bar=fig.add_axes(style['colorbar_axes'])
        cb=fig.colorbar(im,cax=bar,orientation='horizontal',ticks=[0,25,50,75,100])
        cb.ax.xaxis.set_ticks_position('top');cb.ax.tick_params(labelsize=style['colorbar_fontsize']);cb.outline.set_visible(False)
        bx,by,bw,bh=style['colorbar_axes']
        fig.text(bx+bw+.02,by+bh/2,style['colorbar_label'],va='center',fontsize=style['colorbar_fontsize'])
        if style['panel_letter']:
            fig.text(.005,.995,style['panel_letter'],va='top',fontsize=style['panel_fontsize'],fontweight='bold')
        fig.canvas.draw();renderer=fig.canvas.get_renderer()
        for text in ax.get_xticklabels()+ax.get_yticklabels()+list(fig.texts)+[cb.ax.xaxis.label]+cb.ax.get_xticklabels():
            bb=text.get_window_extent(renderer)
            assert bb.x0>=0 and bb.y0>=0 and bb.x1<=fig.bbox.x1 and bb.y1<=fig.bbox.y1,('Clipped text',text.get_text())
        for a,b in zip(ax.get_yticklabels(),ax.get_yticklabels()[1:]):
            assert not a.get_window_extent(renderer).overlaps(b.get_window_extent(renderer)), 'Overlapping node labels'
        for labels in [ax.get_xticklabels(), cb.ax.get_xticklabels()]:
            for left,right in zip(labels,labels[1:]):
                assert not left.get_window_extent(renderer).overlaps(right.get_window_extent(renderer)), 'Overlapping horizontal labels'
        assert all(t.get_fontsize()==style['node_fontsize'] for t in ax.get_xticklabels()+cb.ax.get_xticklabels())
        assert style['colorbar_fontsize']==style['node_fontsize']
        for ext in ['png','svg']:
            fig.savefig(RUN/'figures'/f'{name}.{ext}',dpi=style['dpi'],facecolor='white')
        display(fig);plt.close(fig)

plot_roles(PLOT,'Fig4b_role_stability')
inspection=dict(PLOT,width_inches=8,aspect_ratio=10/17,node_fontsize=12,column_fontsize=12,colorbar_fontsize=12)
plot_roles(inspection,'role_stability_inspection')
original=Image.open(INPUT/'original_figure4.png').convert('RGB')
assert original.size==(4565,3515)
slot=(3021,0,4565,3515)  # Exact panel-b replacement rectangle, leaving the white gutter and panels a/c intact.
panel=Image.open(RUN/'figures'/'Fig4b_role_stability.png').convert('RGB')
assert abs(panel.width/panel.height-(slot[2]-slot[0])/slot[3])<.001
assembled=original.copy()
assembled.paste(panel.resize((1544,3515),Image.Resampling.LANCZOS),(3021,0))
assert np.array_equal(np.asarray(assembled)[:,:3021],np.asarray(original)[:,:3021])
assembled.save(RUN/'figures'/'Figure4_stability.png',dpi=(600,600))
(RUN/'figures'/'layout.json').write_text(json.dumps(dict(original_size=original.size,panel_slot=slot,aspect_ratio=1544/3515,plot=PLOT),indent=2))
validation=dict(complete=True,replicates=1000,original_nodes=38,frequencies_sum_to_100=True,
                baseline_verified=True,deterministic_replicates=[0,499,999],resume_preserves_checkpoint=True,
                inputs_unchanged=True,panels_a_c_pixels_unchanged=True,figure_labels_checked=True,
                scientific_fingerprint=FINGERPRINT)
(RUN/'validation.json').write_text(json.dumps(validation,indent=2))
print('Outputs:',RUN)
